# LangChain: Memory (Modernized)

## Outline
* InMemoryChatMessageHistory (replaces ConversationBufferMemory)
* Windowed memory via `trim_messages` (replaces ConversationBufferWindowMemory)
* Token-limited memory via `trim_messages` (replaces ConversationTokenBufferMemory)
* Summary memory via LLM summarization (replaces ConversationSummaryMemory)

> **Note**: This notebook uses the modern LangChain APIs (`RunnableWithMessageHistory`, `trim_messages`, `InMemoryChatMessageHistory`) instead of the deprecated `ConversationChain` and legacy memory classes.

## InMemoryChatMessageHistory (Buffer Memory)

In [8]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

import warnings
warnings.filterwarnings('ignore')

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

In [9]:
# Set the model variable
llm_model = os.getenv("OPENAI_MODEL", "gpt-4o")

In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import trim_messages

In [11]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

# Session history store (replaces ConversationBufferMemory)
store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain = prompt | llm

conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [12]:
config = {"configurable": {"session_id": "abc123"}}

response = conversation.invoke({"input": "Hi, my name is Andrew"}, config=config)
print(response.content)

Hello, Andrew! How can I assist you today?


In [13]:
response = conversation.invoke({"input": "What is 1+1?"}, config=config)
print(response.content)

1 + 1 equals 2.


In [14]:
response = conversation.invoke({"input": "What is my name?"}, config=config)
print(response.content)

Your name is Andrew.


In [15]:
# View the full conversation history (replaces memory.buffer)
history = get_session_history("abc123")
for msg in history.messages:
    print(f"{msg.type}: {msg.content}")

human: Hi, my name is Andrew
ai: Hello, Andrew! How can I assist you today?
human: What is 1+1?
ai: 1 + 1 equals 2.
human: What is my name?
ai: Your name is Andrew.


In [16]:
# Inspect stored messages (replaces memory.load_memory_variables)
history.messages

[HumanMessage(content='Hi, my name is Andrew', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello, Andrew! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 23, 'total_tokens': 34, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_10ed490cb0', 'id': 'chatcmpl-DSP8ElvTOHys50BJZSHyK3FiF0eZ1', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d6db6-5b2e-7db1-acb9-2e1e6a5d24c6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 23, 'output_tokens': 11, 'total_tokens': 34, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),


In [17]:
# Create a fresh history and manually add messages (replaces memory.save_context)
manual_history = InMemoryChatMessageHistory()

In [18]:
manual_history.add_user_message("Hi")
manual_history.add_ai_message("What's up")

In [19]:
for msg in manual_history.messages:
    print(f"{msg.type}: {msg.content}")

human: Hi
ai: What's up


In [20]:
manual_history.messages

[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}),
 AIMessage(content="What's up", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [21]:
manual_history.add_user_message("Not much, just hanging")
manual_history.add_ai_message("Cool")

In [22]:
manual_history.messages

[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}),
 AIMessage(content="What's up", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Not much, just hanging', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Cool', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## Windowed Memory via `trim_messages` (replaces ConversationBufferWindowMemory)

In [23]:
from langchain_core.messages import HumanMessage, AIMessage

In [24]:
# Manually add messages, then trim to last k exchanges
window_history = InMemoryChatMessageHistory()

In [25]:
window_history.add_user_message("Hi")
window_history.add_ai_message("What's up")
window_history.add_user_message("Not much, just hanging")
window_history.add_ai_message("Cool")

In [26]:
# Keep only the last 2 messages (equivalent to k=1 exchange window)
trimmed = trim_messages(
    window_history.messages,
    max_tokens=2,  # last 2 messages = 1 human-AI exchange
    token_counter=len,  # count by number of messages
    strategy="last",
)
trimmed

[HumanMessage(content='Not much, just hanging', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Cool', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [30]:
# Windowed conversation: only keep last k=1 exchange in context
from langchain_core.runnables import RunnableLambda

llm = ChatOpenAI(temperature=0.0, model=llm_model)

window_store: dict[str, InMemoryChatMessageHistory] = {}

def get_windowed_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in window_store:
        window_store[session_id] = InMemoryChatMessageHistory()
    return window_store[session_id]

windowed_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

def trim_history(input_dict: dict) -> dict:
    """Trim history to last k=1 exchange (2 messages) before sending to prompt."""
    trimmed = trim_messages(
        input_dict["history"],
        max_tokens=2,
        token_counter=len,
        strategy="last",
    )
    return {**input_dict, "history": trimmed}

windowed_chain = RunnableLambda(trim_history) | windowed_prompt | llm

windowed_conversation = RunnableWithMessageHistory(
    windowed_chain,
    get_windowed_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [31]:
window_config = {"configurable": {"session_id": "window123"}}

response = windowed_conversation.invoke({"input": "Hi, my name is Andrew"}, config=window_config)
print(response.content)

Hello, Andrew! How can I assist you today?


In [32]:
response = windowed_conversation.invoke({"input": "What is 1+1?"}, config=window_config)
print(response.content)

1 + 1 equals 2.


In [33]:
# With k=1 window, the model won't remember the name from 2 exchanges ago
response = windowed_conversation.invoke({"input": "What is my name?"}, config=window_config)
print(response.content)

I'm sorry, but I don't have access to personal information about you, including your name. If you'd like, you can tell me your name or ask me anything else!


## Token-Limited Memory via `trim_messages` (replaces ConversationTokenBufferMemory)

In [ ]:
#!pip install tiktoken

In [34]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [35]:
# Build message history and trim by token count (replaces ConversationTokenBufferMemory)
token_history = InMemoryChatMessageHistory()
token_history.add_user_message("AI is what?!")
token_history.add_ai_message("Amazing!")
token_history.add_user_message("Backpropagation is what?")
token_history.add_ai_message("Beautiful!")
token_history.add_user_message("Chatbots are what?")
token_history.add_ai_message("Charming!")

# Trim to ~50 tokens using the LLM's token counter
trimmed_messages = trim_messages(
    token_history.messages,
    max_tokens=50,
    token_counter=llm,
    strategy="last",
)
trimmed_messages

[HumanMessage(content='AI is what?!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Amazing!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Backpropagation is what?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Beautiful!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Chatbots are what?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Charming!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [36]:
# Compare: full history vs trimmed
print(f"Full history: {len(token_history.messages)} messages")
print(f"Trimmed: {len(trimmed_messages)} messages")
for msg in trimmed_messages:
    print(f"  {msg.type}: {msg.content}")

Full history: 6 messages
Trimmed: 6 messages
  human: AI is what?!
  ai: Amazing!
  human: Backpropagation is what?
  ai: Beautiful!
  human: Chatbots are what?
  ai: Charming!


## Summary Memory via LLM Summarization (replaces ConversationSummaryBufferMemory)

In [39]:
from langchain_core.messages import SystemMessage

def summarize_messages(messages: list, llm: ChatOpenAI, max_token_limit: int = 100) -> list:
    """Summarize older messages when history exceeds token limit.

    Keeps recent messages intact and summarizes the rest into a SystemMessage.
    """
    token_count = llm.get_num_tokens_from_messages(messages)
    if token_count <= max_token_limit:
        return messages

    # Find split point: summarize older messages, keep recent ones
    recent_tokens = 0
    split_idx = len(messages)
    for i in range(len(messages) - 1, -1, -1):
        recent_tokens = llm.get_num_tokens_from_messages(messages[i:])
        if recent_tokens > max_token_limit // 2:
            split_idx = i + 1
            break

    old_messages = messages[:split_idx]
    recent_messages = messages[split_idx:]

    # Summarize old messages
    summary_prompt = (
        "Distill the above chat messages into a single summary message. "
        "Include as many specific details as you can."
    )
    summary_messages = old_messages + [HumanMessage(content=summary_prompt)]
    summary = llm.invoke(summary_messages)

    return [SystemMessage(content=f"Summary of earlier conversation: {summary.content}")] + recent_messages

In [37]:
# Create a long string
schedule = "There is a meeting at 8am with your product team. \
You will need your powerpoint presentation prepared. \
9am-12pm have time to work on your LangChain \
project which will go quickly because Langchain is such a powerful tool. \
At Noon, lunch at the italian resturant with a customer who is driving \
from over an hour away to meet you to understand the latest in AI. \
Be sure to bring your laptop to show the latest LLM demo."

# Build up a conversation history
summary_history = InMemoryChatMessageHistory()
summary_history.add_user_message("Hello")
summary_history.add_ai_message("What's up")
summary_history.add_user_message("Not much, just hanging")
summary_history.add_ai_message("Cool")
summary_history.add_user_message("What is on the schedule today?")
summary_history.add_ai_message(schedule)

In [40]:
# Summarize when history gets too long (max 100 tokens)
summarized = summarize_messages(summary_history.messages, llm, max_token_limit=100)
for msg in summarized:
    print(f"{msg.type}: {msg.content[:120]}...")

system: Summary of earlier conversation: Today, you have an 8am meeting with your product team, requiring a prepared PowerPoint ...


In [41]:
# Use the summarized history in a conversation
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# Pre-populate a session with the summarized history
summary_store: dict[str, InMemoryChatMessageHistory] = {}

def get_summary_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in summary_store:
        h = InMemoryChatMessageHistory()
        # Seed with summarized messages
        for msg in summarized:
            h.add_message(msg)
        summary_store[session_id] = h
    return summary_store[session_id]

summary_chain = summary_prompt | llm
summary_conversation = RunnableWithMessageHistory(
    summary_chain,
    get_summary_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [42]:
summary_config = {"configurable": {"session_id": "summary123"}}

response = summary_conversation.invoke(
    {"input": "What would be a good demo to show?"},
    config=summary_config,
)
print(response.content)

For your lunch meeting demo, you could showcase a few key features of the latest LLM that would be impressive and relevant to your customer. Here are some ideas:

1. **Real-time Text Generation**: Demonstrate the model's ability to generate coherent and contextually relevant text in real-time. You could start with a prompt related to the customer's industry and show how the model can expand on it.

2. **Question Answering**: Show how the LLM can answer complex questions accurately. You could use a dataset or questions that are specific to the customer's field to make it more relevant.

3. **Summarization**: Present the model's ability to summarize long documents or articles. This could be particularly useful if the customer deals with large volumes of text data.

4. **Conversational AI**: If applicable, demonstrate a chatbot or virtual assistant powered by the LLM. Highlight its ability to understand context and maintain a natural conversation flow.

5. **Customizable Outputs**: Show h

In [43]:
# View the final conversation history including summary
final_history = get_summary_history("summary123")
for msg in final_history.messages:
    print(f"{msg.type}: {msg.content[:150]}")

system: Summary of earlier conversation: Today, you have an 8am meeting with your product team, requiring a prepared PowerPoint presentation. From 9am to 12pm
human: What would be a good demo to show?
ai: For your lunch meeting demo, you could showcase a few key features of the latest LLM that would be impressive and relevant to your customer. Here are 
